In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range
from rank_bm25 import BM25Okapi
import re
import json

load_dotenv()

PROCESSED_DATA = Path("../data/processed")
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
QDRANT_URL = "http://localhost:6333"

model = SentenceTransformer(EMBEDDING_MODEL)
client = QdrantClient(url=QDRANT_URL)

print(f"Collections: {[c.name for c in client.get_collections().collections]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collections: ['course_profiles', 'rider_seasons']


In [ ]:
merged_df = pd.read_csv(PROCESSED_DATA / "merged_uci_races.csv", low_memory=False)
course_data = pd.read_csv(PROCESSED_DATA / "course_data_clean.csv")

print(f"merged_df:   {merged_df.shape[0]:,} rows")
print(f"course_data: {course_data.shape[0]:,} rows")

merged_df:   140,573 rows
course_data: 963 rows


In [3]:
def tokenize(text: str) -> list[str]:
    """Simple tokenizer — lowercase, remove punctuation, split on whitespace."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

# Rebuild course docs (same serializer logic as notebook 2)
def serialize_course_doc(row: pd.Series) -> str:
    parts = []
    parts.append(f"Race: {row['Race Name']}.")
    if pd.notna(row.get('Distance')):
        parts.append(f"Distance: {row['Distance']:.1f} km.")
    if pd.notna(row.get('Vertical Gain')) and pd.notna(row.get('Highest Elevation')):
        vg = row['Vertical Gain']
        he = row['Highest Elevation']
        le = row.get('Lowest Elevation', 0)
        ng = row.get('Net Gain', 0)
        if vg > 4000:
            stage_type = "a high mountain stage"
        elif vg > 2000:
            stage_type = "a hilly stage with significant climbing"
        elif vg > 800:
            stage_type = "a moderately hilly stage"
        else:
            stage_type = "a flat stage suited to sprinters"
        parts.append(
            f"This is {stage_type} with {vg:.0f}m of vertical gain, "
            f"reaching a maximum elevation of {he:.0f}m "
            f"and a minimum elevation of {le:.0f}m. "
            f"Net elevation change: {ng:.0f}m."
        )
    surfaces = {
        'Asphalt': 'asphalt', 'Road': 'road',
        'Cobblestones': 'cobblestones', 'Compacted Gravel': 'compacted gravel',
        'Unpaved': 'unpaved sections', 'Paved': 'paved road',
    }
    surface_parts = []
    for col, label in surfaces.items():
        val = row.get(col)
        if pd.notna(val) and float(val) > 0.5:
            surface_parts.append(f"{val:.1f}km of {label}")
    if surface_parts:
        parts.append(f"Surface breakdown: {', '.join(surface_parts)}.")
    cob = row.get('Cobblestones')
    if pd.notna(cob) and float(cob) > 0:
        parts.append(
            f"Contains {cob:.1f}km of cobblestones — "
            f"a significant factor for classics specialists."
        )
    dh = row.get('Downhill')
    if pd.notna(dh) and float(dh) > 0:
        parts.append(f"Total descending: {dh:.0f}m.")
    return " ".join(parts)

def serialize_rider_doc(rider_name: str, year: int, df: pd.DataFrame) -> str:
    rider_df = df[
        (df['Name'] == rider_name) &
        (df['Year_results'] == year)
    ].copy()
    if rider_df.empty:
        return ""
    parts = []
    parts.append(f"Rider: {rider_name}. Season: {year}.")
    team = rider_df['Team'].iloc[0]
    if pd.notna(team):
        parts.append(f"Team: {team}.")
    races = rider_df['Race_results'].unique()
    parts.append(f"Competed in {len(races)} races.")
    wins = rider_df[rider_df['Rank'] == 1]
    podiums = rider_df[rider_df['Top3'] == 1]
    top10s = rider_df[rider_df['Top10'] == 1]
    dnfs = rider_df[~rider_df['Did_Finish']]
    parts.append(
        f"Results: {len(wins)} wins, {len(podiums)} podiums, "
        f"{len(top10s)} top-10 finishes, {len(dnfs)} DNFs/DNS."
    )
    if not wins.empty:
        win_names = wins['Race Name'].tolist()
        parts.append(f"Won: {', '.join(win_names[:5])}.")
    top_results = rider_df[
        (rider_df['Rank'] > 1) & (rider_df['Did_Finish'])
    ].nsmallest(3, 'Rank')
    if not top_results.empty:
        best = [
            f"{row['Race Name']} (P{int(row['Rank'])})"
            for _, row in top_results.iterrows()
        ]
        parts.append(f"Other notable results: {', '.join(best)}.")
    gc_races = ['Tour de France', 'Giro d', 'Vuelta a Espana']
    for gc in gc_races:
        gc_results = rider_df[rider_df['Race_results'].str.contains(gc, na=False)]
        if not gc_results.empty:
            best_gc = gc_results[gc_results['Did_Finish']]['Rank'].min()
            if best_gc < 999:
                parts.append(f"{gc}: best result P{int(best_gc)}.")
    return " ".join(parts)

# Build course corpus
print("Building course corpus...")
course_corpus = []
course_ids = []
for _, row in course_data.iterrows():
    text = serialize_course_doc(row)
    if text.strip():
        course_corpus.append(text)
        course_ids.append(row['Race Name'])

# Build rider corpus
print("Building rider corpus...")
rider_corpus = []
rider_ids = []
rider_seasons = merged_df.groupby(['Name', 'Year_results']).size().reset_index()
for _, row in rider_seasons.iterrows():
    text = serialize_rider_doc(row['Name'], row['Year_results'], merged_df)
    if text.strip():
        rider_corpus.append(text)
        rider_ids.append(f"{row['Name']}_{int(row['Year_results'])}")

print(f"Course corpus: {len(course_corpus):,} documents")
print(f"Rider corpus:  {len(rider_corpus):,} documents")

Building course corpus...
Building rider corpus...
Course corpus: 963 documents
Rider corpus:  5,835 documents


In [4]:
print("Building BM25 indexes...")

course_tokenized = [tokenize(doc) for doc in course_corpus]
rider_tokenized  = [tokenize(doc) for doc in rider_corpus]

bm25_course = BM25Okapi(course_tokenized)
bm25_rider  = BM25Okapi(rider_tokenized)

print(f"BM25 course index: {len(course_tokenized):,} documents")
print(f"BM25 rider index:  {len(rider_tokenized):,} documents")

Building BM25 indexes...
BM25 course index: 963 documents
BM25 rider index:  5,835 documents


In [5]:
def semantic_search(
    query: str,
    collection: str,
    top_k: int = 20,
    filter_condition=None
) -> list[dict]:
    query_vector = model.encode(query).tolist()
    results = client.query_points(
        collection_name=collection,
        query=query_vector,
        limit=top_k,
        with_payload=True,
        query_filter=filter_condition
    )
    return [
        {
            "id": r.payload.get("doc_id"),
            "score": r.score,
            "text": r.payload.get("text", ""),
            "payload": r.payload
        }
        for r in results.points
    ]

In [6]:
def bm25_search(
    query: str,
    corpus: list[str],
    ids: list[str],
    bm25_index: BM25Okapi,
    top_k: int = 20
) -> list[dict]:
    tokens = tokenize(query)
    scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "id": ids[i],
            "score": float(scores[i]),
            "text": corpus[i]
        }
        for i in top_indices
        if scores[i] > 0  # only return docs with non-zero BM25 score
    ]

In [7]:
def reciprocal_rank_fusion(
    result_lists: list[list[dict]],
    k: int = 60,
    top_k: int = 5
) -> list[dict]:
    """
    Combine multiple ranked result lists using Reciprocal Rank Fusion.
    RRF score = sum(1 / (k + rank)) across all lists.
    k=60 is the standard default from the original RRF paper.
    """
    scores = {}
    doc_text = {}

    for result_list in result_lists:
        for rank, doc in enumerate(result_list, start=1):
            doc_id = doc["id"]
            if doc_id not in scores:
                scores[doc_id] = 0.0
                doc_text[doc_id] = doc["text"]
            scores[doc_id] += 1.0 / (k + rank)

    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [
        {
            "id": doc_id,
            "rrf_score": round(score, 6),
            "text": doc_text[doc_id]
        }
        for doc_id, score in sorted_docs[:top_k]
    ]

def hybrid_search(
    query: str,
    collection: str,
    corpus: list[str],
    ids: list[str],
    bm25_index: BM25Okapi,
    top_k: int = 5,
    fetch_k: int = 20
) -> list[dict]:
    """
    Full hybrid search pipeline:
    1. Semantic search via Qdrant
    2. Lexical search via BM25
    3. Combine with Reciprocal Rank Fusion
    """
    semantic_results = semantic_search(query, collection, top_k=fetch_k)
    lexical_results  = bm25_search(query, corpus, ids, bm25_index, top_k=fetch_k)
    fused            = reciprocal_rank_fusion([semantic_results, lexical_results], top_k=top_k)
    return fused

In [8]:
course_queries = [
    "What are the hardest mountain stages in the Tour de France?",
    "Cobblestone classics races like Paris-Roubaix",
    "Flat sprint stages with low elevation",
    "Short time trial stages",
    "Strade Bianche gravel parcours",
]

for query in course_queries:
    print(f"\n{'='*65}")
    print(f"Query: '{query}'")
    print("-" * 65)

    results = hybrid_search(
        query,
        collection="course_profiles",
        corpus=course_corpus,
        ids=course_ids,
        bm25_index=bm25_course,
        top_k=5
    )

    for r in results:
        print(f"  [{r['rrf_score']:.5f}] {r['id']}")
        print(f"           {r['text'][:150]}...")
        print()


Query: 'What are the hardest mountain stages in the Tour de France?'
-----------------------------------------------------------------
  [0.03128] 2021 Tour de France Stage 9
           Race: 2021 Tour de France Stage 9. Distance: 144.0 km. This is a high mountain stage with 4600m of vertical gain, reaching a maximum elevation of 2119...

  [0.03080] 2021 Tour de France Stage 17
           Race: 2021 Tour de France Stage 17. Distance: 178.0 km. This is a high mountain stage with 4330m of vertical gain, reaching a maximum elevation of 220...

  [0.02986] 2021 Tour de France Stage 11
           Race: 2021 Tour de France Stage 11. Distance: 199.0 km. This is a high mountain stage with 4710m of vertical gain, reaching a maximum elevation of 189...

  [0.02964] 2022 Tour de France Stage 18
           Race: 2022 Tour de France Stage 18. Distance: 144.0 km. This is a high mountain stage with 4200m of vertical gain, reaching a maximum elevation of 170...

  [0.02822] 2023 Tour de France Stage

In [9]:
rider_queries = [
    "Who won the Tour de France in 2023?",
    "Best climbers Grand Tour winners 2022",
    "Remco Evenepoel 2023 season results",
    "Sprinters who won stages in 2023",
    "Riders with most wins in 2022",
]

for query in rider_queries:
    print(f"\n{'='*65}")
    print(f"Query: '{query}'")
    print("-" * 65)

    results = hybrid_search(
        query,
        collection="rider_seasons",
        corpus=rider_corpus,
        ids=rider_ids,
        bm25_index=bm25_rider,
        top_k=5
    )

    for r in results:
        print(f"  [{r['rrf_score']:.5f}] {r['id']}")
        print(f"           {r['text'][:150]}...")
        print()


Query: 'Who won the Tour de France in 2023?'
-----------------------------------------------------------------
  [0.03033] LAFAY Victor_2023
           Rider: LAFAY Victor. Season: 2023. Team: Cofidis. Competed in 5 races. Results: 1 wins, 1 podiums, 4 top-10 finishes, 3 DNFs/DNS. Won: 2023 Tour de Fr...

  [0.03028] PHILIPSEN Jasper_2023
           Rider: PHILIPSEN Jasper. Season: 2023. Team: Alpecin-Deceuninck. Competed in 6 races. Results: 7 wins, 11 podiums, 13 top-10 finishes, 1 DNFs/DNS. Won...

  [0.02944] MEEUS Jordi_2023
           Rider: MEEUS Jordi. Season: 2023. Team: BORA - hansgrohe. Competed in 6 races. Results: 1 wins, 1 podiums, 8 top-10 finishes, 2 DNFs/DNS. Won: 2023 To...

  [0.02659] VAN DER POEL Mathieu_2023
           Rider: VAN DER POEL Mathieu. Season: 2023. Team: Alpecin-Deceuninck. Competed in 6 races. Results: 1 wins, 2 podiums, 2 top-10 finishes, 0 DNFs/DNS. W...

  [0.01639] BILBAO Pello_2023
           Rider: BILBAO Pello. Season: 2023. Team: Bahrain - V

In [10]:
# This is the key comparison — shows the interviewer you understand
# WHY hybrid search exists, not just how to implement it

comparison_queries = [
    ("Who won the 2023 Tour de France?", "rider_seasons", rider_corpus, rider_ids, bm25_rider),
    ("Paris-Roubaix cobblestone stage profile", "course_profiles", course_corpus, course_ids, bm25_course),
]

for query, collection, corpus, ids, bm25 in comparison_queries:
    print(f"\n{'='*65}")
    print(f"QUERY: '{query}'")

    sem = semantic_search(query, collection, top_k=3)
    hyb = hybrid_search(query, collection, corpus, ids, bm25, top_k=3)

    print(f"\n--- Semantic only ---")
    for r in sem:
        print(f"  [{r['score']:.4f}] {r['id']}")

    print(f"\n--- Hybrid (RRF) ---")
    for r in hyb:
        print(f"  [{r['rrf_score']:.5f}] {r['id']}")


QUERY: 'Who won the 2023 Tour de France?'

--- Semantic only ---
  [0.7241] PHILIPSEN Jasper_2023
  [0.7217] HOULE Hugo_2023
  [0.7201] LAFAY Victor_2023

--- Hybrid (RRF) ---
  [0.03110] PHILIPSEN Jasper_2023
  [0.03016] LAFAY Victor_2023
  [0.02924] MEEUS Jordi_2023

QUERY: 'Paris-Roubaix cobblestone stage profile'

--- Semantic only ---
  [0.6516] 2018 Paris - Roubaix
  [0.6435] 2019 Paris-Roubaix
  [0.6373] 2023 Paris-Roubaix

--- Hybrid (RRF) ---
  [0.03178] 2018 Paris - Roubaix
  [0.03178] 2021 Paris-Roubaix
  [0.03150] 2023 Paris-Roubaix


In [11]:
def structured_search(
    collection: str,
    year: int = None,
    min_vertical_gain: float = None,
    doc_type: str = None,
    top_k: int = 5
) -> list[dict]:
    """
    Query Qdrant using structured metadata filters.
    No embedding needed — pure field matching.
    """
    conditions = []

    if year:
        conditions.append(
            FieldCondition(key="year", match=MatchValue(value=year))
        )
    if min_vertical_gain:
        conditions.append(
            FieldCondition(
                key="vertical_gain",
                range=Range(gte=min_vertical_gain)
            )
        )
    if doc_type:
        conditions.append(
            FieldCondition(key="doc_type", match=MatchValue(value=doc_type))
        )

    scroll_filter = Filter(must=conditions) if conditions else None

    results, _ = client.scroll(
        collection_name=collection,
        scroll_filter=scroll_filter,
        limit=top_k,
        with_payload=True
    )

    return [
        {
            "id": r.payload.get("doc_id"),
            "vertical_gain": r.payload.get("vertical_gain"),
            "text": r.payload.get("text", "")[:200]
        }
        for r in results
    ]

# Test: find all 2023 stages with >4000m vertical gain
print("=== 2023 stages with >4000m vertical gain ===")
hard_stages = structured_search(
    "course_profiles",
    year=2023,
    min_vertical_gain=4000,
    top_k=10
)
for r in hard_stages:
    print(f"  {r['id']} — {r['vertical_gain']}m vertical gain")

=== 2023 stages with >4000m vertical gain ===
  2023 Criterium du Dauphine Stage 7 — 4120.0m vertical gain
  2023 Criterium du Dauphine Stage 8 — 4140.0m vertical gain
  2023 Giro dItalia Stage 15 — 4250.0m vertical gain
  2023 Giro dItalia Stage 16 — 6450.0m vertical gain
  2023 Giro dItalia Stage 18 — 4200.0m vertical gain
  2023 Giro dItalia Stage 19 — 5560.0m vertical gain
  2023 Giro dItalia Stage 7 — 4170.0m vertical gain
  2023 Il Lombardia — 5070.0m vertical gain
  2023 La Vuelta Ciclista a Espana Stage 13 — 4719.9999999999m vertical gain
  2023 La Vuelta Ciclista a Espana Stage 14 — 4310.0m vertical gain
